# 📝 Google Colab: ทดสอบตรวจกระดาษคำตอบจริงด้วยโมเดล .pt
**โครงการ:** ระบบตรวจกระดาษคำตอบและให้คะแนนอัตโนมัติ Rayong MARCO (Template 2)  
**สถาบัน:** มหาวิทยาลัยบูรพา  
---
สมุดงาน (Notebook) นี้ใช้สำหรับ **นำโมเดล `.pt` ที่เทรนเสร็จแล้วมาทดสอบตรวจกระดาษคำตอบจริง**:
1. 📥 โหลดโมเดล PyTorch `.pt` ครบทั้ง 3 ตัว (HCR A-Z, Strikethrough `=`, Header Digits 0-9)
2. 📸 **อัปโหลดภาพถ่ายกระดาษคำตอบของคุณเอง** หรือเลือกจากชุดภาพทดสอบที่มีอยู่
3. ⚙️ รันไปป์ไลน์ตรวจให้คะแนนอัตโนมัติครบ 5 Phase ในเวลา ~2.4 วินาที
4. 🖼️ **แสดงภาพเฉลยพร้อมกรอบตรวจคะแนน (Audit Overlay)** ใน Colab ทันที
5. 💾 ดาวน์โหลดภาพผลการตรวจและไฟล์คะแนนกลับสู่คอมพิวเตอร์

### ⚙️ ขั้นตอนที่ 1: ดึงโค้ดและโมเดล `.pt` จาก GitHub

In [ ]:
# 1. Clone โปรเจกต์และติดตั้งไลบรารี
!git clone https://github.com/67160328/Computer-Vision.git
%cd Computer-Vision/Lab9
!pip install -q opencv-python numpy pandas matplotlib tqdm

# ตรวจสอบว่ามีไฟล์โมเดล .pt ครบทั้ง 3 ตัว
!ls -lh models/

### 🧠 ขั้นตอนที่ 2: โหลดเอนจินและโมเดลทั้ง 3 ตัวเข้าสู่หน่วยความจำ

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.grading.grader import AutomatedGradingEngine
from src.evaluation.evaluate_phase3 import imread_bgr_unicode, imwrite_unicode

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 กำลังใช้งานประมวลผลบน: {device.upper()}")

# โหลดโมเดล .pt ทั้ง 3 ตัว
grader = AutomatedGradingEngine(
    hcr_weights_path="models/hcr_resnet28_az.pt",
    strikethrough_weights_path="models/strikethrough_cnn.pt",
    digit_weights_path="models/hcr_resnet28_digits.pt",
    device=device,
)
print("✅ โหลดโมเดล .pt ทั้ง 3 ตัวสำเร็จ พร้อมตรวจข้อสอบแล้ว!")

### 🔑 ขั้นตอนที่ 3: กำหนดชุดเฉลยคำตอบ (Answer Key สำหรับข้อ 1 – 20)
*(คุณสามารถแก้ไขคำในตัวแปร `CUSTOM_ANSWER_KEY` ให้ตรงกับข้อสอบของคุณได้เลยครับ)*

In [ ]:
# เฉลยคำตอบข้อ 1 - 20 (สามารถแก้ไขเป็นคำศัพท์ที่คุณต้องการได้)
CUSTOM_ANSWER_KEY = {
    1: "ELECTROENCEPHALOGRAM",  # 20 ตัวอักษร (เต็ม 20 ช่อง)
    2: "UNCHARACTERISTICALLY",  # 20 ตัวอักษร (เต็ม 20 ช่อง)
    3: "COUNTERREVOLUTIONARY",  # 20 ตัวอักษร (เต็ม 20 ช่อง)
    4: "INCOMPREHENSIBILITY",   # 19 ตัวอักษร
    5: "DISPROPORTIONATELY",    # 18 ตัวอักษร
    6: "PSYCHOPHARMACOLOGY",    # 18 ตัวอักษร
    7: "TRANSUBSTANTIATION",    # 18 ตัวอักษร
    8: "OVERSIMPLIFICATION",    # 18 ตัวอักษร
    9: "CONSTITUTIONALITY",     # 17 ตัวอักษร
    10: "TELECOMMUNICATION",    # 17 ตัวอักษร
    11: "ENVIRONMENTALISTS",    # 17 ตัวอักษร
    12: "STRAIGHTFORWARDLY",    # 17 ตัวอักษร
    13: "INDIVIDUALIZATION",    # 17 ตัวอักษร
    14: "MISUNDERSTANDINGS",    # 17 ตัวอักษร
    15: "DISQUALIFICATION",     # 16 ตัวอักษร
    16: "EXTRATERRESTRIAL",     # 16 ตัวอักษร
    17: "BIOTECHNOLOGICAL",     # 16 ตัวอักษร
    18: "ACKNOWLEDGEMENTS",     # 16 ตัวอักษร
    19: "JURISDICTIONAL",       # 14 ตัวอักษร
    20: "CYBERSECURITY",         # 13 ตัวอักษร
}
print(f"✅ บันทึกชุดเฉลย {len(CUSTOM_ANSWER_KEY)} ข้อเรียบร้อย")

### 📸 ขั้นตอนที่ 4: เลือกรูปกระดาษคำตอบที่จะตรวจ
เลือกวิธีใดวิธีหนึ่ง:
* **ทางเลือก A:** รัน Cell 4A เพื่อ **อัปโหลดภาพถ่ายของคุณเอง** จากมือถือหรือคอมพิวเตอร์
* **ทางเลือก B:** รัน Cell 4B เพื่อ **ใช้รูปตัวอย่างที่มีอยู่แล้วในระบบ**

In [ ]:
# [ทางเลือก A] อัปโหลดภาพของคุณเองจากคอมพิวเตอร์
from google.colab import files

print("กรุณาเลือกไฟล์ภาพถ่ายกระดาษคำตอบของคุณ (jpg, png):")
uploaded = files.upload()
if uploaded:
    TARGET_IMAGE_PATH = list(uploaded.keys())[0]
    print(f"✅ อัปโหลดไฟล์สำเร็จ: {TARGET_IMAGE_PATH}")
else:
    print("ไม่ได้เลือกไฟล์")

In [ ]:
# [ทางเลือก B] หรือจะใช้ภาพตัวอย่างที่มีในโปรเจกต์ (ปลดคอมเมนต์ภาพที่ต้องการ)
TARGET_IMAGE_PATH = "data/Test/ตัวอย่างที่ 3.jpg"
# TARGET_IMAGE_PATH = "data/Test/ตัวอย่างที่ 4.jpg"
# TARGET_IMAGE_PATH = "data/Test/ตัวอย่างการขีดฆ่า.jpg"
# TARGET_IMAGE_PATH = "data/dataset_splits/sheets_raw/test/02_edge_strikethrough/02_edge_strikethrough_005.jpg"

print(f"เลือกตรวจภาพ: {TARGET_IMAGE_PATH}")

### ⚡ ขั้นตอนที่ 5: ประมวลผลและตรวจให้คะแนน (End-to-End Grading)

In [ ]:
# โหลดภาพและรันการตรวจ
img_bgr = imread_bgr_unicode(Path(TARGET_IMAGE_PATH))
assert img_bgr is not None, f"ไม่สามารถเปิดไฟล์ {TARGET_IMAGE_PATH} ได้"

print("⏳ กำลังประมวลผลกระดาษคำตอบ...")
report, res1, grid, state_map, pre_char_map = grader.grade_sheet(
    img_bgr=img_bgr,
    image_name=Path(TARGET_IMAGE_PATH).name,
    answer_key=CUSTOM_ANSWER_KEY,
    category="colab_test",
)

# แสดงผลลัพธ์ส่วนหัวกระดาษ
print("\n" + "=" * 60)
print(" 📌 ข้อมูลที่สแกนได้จากส่วนหัว (Header ID Strip 42 ช่องสีม่วง)")
print("=" * 60)
print(f"  • รหัสประจำตัว (Student ID) : {report.student_id or '(เว้นว่าง)'}")
print(f"  • รหัสวิชา (Subject ID)     : {report.subject_id or '(เว้นว่าง)'}")
print(f"  • รหัสสถานศึกษา (School ID) : {report.school_id or '(เว้นว่าง)'}")
print(f"  • ชุดข้อสอบ                 : {report.detected_question_set or '-'}")
print(f"  • หน้ากระดาษ               : {report.detected_page_number or '-'}")

# แสดงตารางผลคะแนน 20 ข้อ
print("\n" + "=" * 60)
print(" 📝 ตารางผลคะแนนรายข้อ (ข้อ 1 - 20)")
print("=" * 60)
print(f"{'ข้อ':^4} | {'คำที่อ่านได้':<18} | {'เฉลย':<18} | {'สถานะ':^8} | {'ขีดฆ่า (=)':^10}")
print("-" * 68)
for q in report.questions:
    if q.ground_truth_word:
        status = "✅ PASS" if q.is_correct else "❌ FAIL"
        if q.needs_human_review:
            status += " ⚠️"
    else:
        status = "⚪ NO KEY"
    strike_str = f"{q.strikethrough_count} ช่อง" if q.strikethrough_count > 0 else "-"
    print(f"{q.question_num:^4} | {q.final_assembled_word:<18} | {q.ground_truth_word:<18} | {status:^8} | {strike_str:^10}")

print("-" * 68)
print(f" 🏆 คะแนนรวมสุทธิ : {report.total_score} / {report.max_possible_score} คะแนน")
print(f" 🎯 ความแม่นยำระดับคำ : {report.exact_match_accuracy_pct:.2f}%")
print(f" ⏱️ เวลาตรวจทั้งแผ่น : {report.total_pipeline_ms:.1f} ms (~{report.total_pipeline_ms/1000.0:.2f} วินาที)")

### 🖼️ ขั้นตอนที่ 6: แสดงภาพผลการตรวจข้อสอบพร้อม Overlay ใน Colab

In [ ]:
# สร้างภาพผลการตรวจพร้อมป้ายคะแนนและกรอบสี
vis = grader.render_graded_sheet_overlay(
    res1.rectified_bgr,
    grid,
    state_map,
    pre_char_map,
    report,
    header_results=getattr(grader, "_last_header_results", None),
)

# บันทึกไฟล์ภาพ
output_jpg = "graded_result_colab.jpg"
imwrite_unicode(Path(output_jpg), vis)

# แสดงภาพผลตรวจใน Colab ขนาดใหญ่
vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 22))
plt.imshow(vis_rgb)
plt.axis("off")
plt.title(f"Graded Result: {report.total_score}/{report.max_possible_score} Points ({report.exact_match_accuracy_pct:.1f}%)", fontsize=16)
plt.show()

### 💾 ขั้นตอนที่ 7: ดาวน์โหลดภาพผลการตรวจกลับเครื่อง

In [ ]:
from google.colab import files

print("กำลังดาวน์โหลดภาพผลการตรวจ...")
files.download("graded_result_colab.jpg")